In [0]:
from pyspark.sql.functions import (
    col, initcap, regexp_replace, to_date, coalesce, 
    current_timestamp, expr, trim
)
import re

spark.sql("USE CATALOG 02_silver")
spark.sql("USE SCHEMA default")

def clean_column(name):
    name = re.sub(r'[^a-zA-Z0-9]', '', name)
    return name.lower()

In [0]:
df = spark.read.table("01_bronze.default.raw_sales_dtl")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_sales = df.select(
    col("trxnid").alias("transaction_id"),
    col("custid99").alias("customer_id"),
    col("storelocid").alias("store_id"),
    col("prodcodeid").alias("product_id"),
    
    to_date(
        coalesce(
            expr("try_to_timestamp(dateref, 'dd-MM-yyyy')"),
            expr("try_to_timestamp(dateref, 'MMM dd, yyyy')"),
            expr("try_to_timestamp(dateref, 'yyyy-MM-dd')")
        )
    ).alias("transaction_date"),
    
    col("qtysold").cast("int").alias("quantity"),
    
    regexp_replace(col("unitprice"), "[^0-9.]", "").cast("double").alias("selling_price")
)
df_sales = df_sales.withColumn("cost_price", col("selling_price") * 0.7)
df_sales = df_sales.withColumn("discount", col("selling_price") * 0.1)

df_sales = df_sales.withColumn("ingested_at", current_timestamp())

print("Row count:", df_sales.count())
df_sales.printSchema()
spark.sql("DROP TABLE IF EXISTS 02_silver.default.sales")

df_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("02_silver.default.sales")

spark.sql("SELECT * FROM 02_silver.default.sales LIMIT 10").show()

In [0]:
df = spark.read.table("01_bronze.default.mst_product_master")
df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_products = df.select(
    col("productid").alias("product_id"),
    initcap(
        trim(regexp_replace(col("itemnamedescription"), "[^a-zA-Z\\s]", ""))
    ).alias("product_name"),
    
    initcap(col("categorytype")).alias("category"),
    regexp_replace(col("basecost"), "[^0-9.]", "").cast("double").alias("cost_price")
)

df_products = df_products.fillna({
    "product_id": "unknown",
    "product_name": "Unknown",
    "category": "Unknown",
    "cost_price": 0
})

df_products = df_products.dropDuplicates(["product_id"])

df_products = df_products.withColumn("ingested_at", current_timestamp())

spark.sql("DROP TABLE IF EXISTS products")

df_products.write.format("delta").mode("overwrite").saveAsTable("products")

In [0]:
df = spark.read.table("01_bronze.default.inv_levels_raw")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_inventory = df.select(
    col("skuid").alias("product_id"),
    col("stidref").alias("store_id"),
    
    col("stockonhand").cast("int").alias("stock_qty")
)

df_inventory = df_inventory.fillna({
    "stock_qty": 0
})

df_inventory = df_inventory.dropDuplicates()

df_inventory = df_inventory.withColumn("ingested_at", current_timestamp())

spark.sql("DROP TABLE IF EXISTS inventory")

df_inventory.write.format("delta").mode("overwrite").saveAsTable("inventory")

In [0]:
df = spark.read.table("01_bronze.default.rtn_trans_logs")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_returns = df.select(
    col("orgnltrxnid").alias("transaction_id"),
    col("rtnidno").alias("return_id"),
    
    to_date(
        coalesce(
            expr("try_to_timestamp(rtndatestring, 'dd-MM-yyyy')"),
            expr("try_to_timestamp(rtndatestring, 'MMM dd, yyyy')"),
            expr("try_to_timestamp(rtndatestring, 'yyyy-MM-dd')")
        )
    ).alias("return_date"),
    
    col("rsncode").alias("return_reason")
)

df_returns = df_returns.fillna({
    "return_reason": "unknown"
})

df_returns = df_returns.dropDuplicates(["return_id"])

df_returns = df_returns.withColumn("ingested_at", current_timestamp())

spark.sql("DROP TABLE IF EXISTS returns")

df_returns.write.format("delta").mode("overwrite").saveAsTable("returns")

In [0]:
df = spark.read.table("01_bronze.default.stores_geo_list_final")

df = df.select([col(c).alias(clean_column(c)) for c in df.columns])

df_stores = df.select(
    col("stid").alias("store_id"),
    
    # clean name (remove special chars)
    initcap(
        trim(regexp_replace(col("locationnameaddress"), "[^a-zA-Z\\s]", ""))
    ).alias("store_name"),
    
    initcap(col("cityregion")).alias("city"),
    
    # optional (can keep or drop)
    regexp_replace(col("mgrcontact"), "[^0-9]", "").alias("manager_contact")
)

df_stores = df_stores.fillna({
    "store_id": "unknown",
    "store_name": "Unknown",
    "city": "Unknown",
    "manager_contact": "0000000000"
})

df_stores = df_stores.dropDuplicates(["store_id"])

df_stores = df_stores.withColumn("ingested_at", current_timestamp())

spark.sql("DROP TABLE IF EXISTS stores")

df_stores.write.format("delta").mode("overwrite").saveAsTable("stores")